In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


📌 **Introduction**

This project analyzes and forecasts my personal spending using data extracted from Google Pay through Google Takeout. The raw transaction history was provided in HTML format, so I first used Python to convert it into a clean CSV file for analysis. After preprocessing the data—keeping only completed outgoing transactions and aggregating it by date—I explored whether time-series models could predict future spending. The goal was to build and test an LSTM model, and later compare it with Prophet/NeuralProphet to understand how well different approaches can handle real-world, irregular financial data.

In [ ]:
df = pd.read_csv(r'/content/extracted_gpay_data.csv')
df

In [ ]:
df.columns

In [ ]:
df.status.value_counts()

In [ ]:
#taking only the completed transactions into consideration


# Standardizing cause the data can be like (Completed or COMPLETED etc)
df['status'] = df['status'].astype(str).str.strip().str.lower()


df = df[df['status'] == 'completed'].reset_index(drop=True)


print(df['status'].value_counts())



In [ ]:
#keeping only the expense transaction
df['transaction_type'] = df['transaction_type'].astype(str).str.strip().str.lower()

# Keep only outgoing money transactions
df = df[df['transaction_type'].isin(['sent', 'paid'])]

print(df['transaction_type'].value_counts())


In [ ]:
#analyzing the data
df.describe()

**Insights :**

* My Avg spending = 505
* My Max amount spent at once was 80000 & Min = 1
* 75% of My transactions are below ₹120
* 50% below ₹30

In [ ]:
df.dtypes

In [ ]:
#converting the date datatype into datetime
df['date']=pd.to_datetime(df['date'])
df.dtypes

In [ ]:
df['date'].min(),df['date'].max()  #ie from 14/03/2022 to 13/08/2025

**Preprocessing for the LSTM model**

In [ ]:
#Step 1 :- Single dimensional data
df = df[["date","amount_inr"]]
df

In [ ]:
#Step 2 :- sorting the data in asc order using the date col

df=df.sort_values('date')
df.isnull().sum()

In [ ]:
#Step 3 :- there should be only one value in one date

df.date.value_counts()

In [ ]:
df=df.groupby('date')['amount_inr'].sum().reset_index()  #we basically added all the transactions belonging to one date
df.date.value_counts()

In [ ]:
#step 4 :- set the index to  date

df=df.set_index('date')


In [ ]:
df.head()

In [ ]:
#Steap 5 :- Resampling

y = df['amount_inr'].resample("ME").sum()

In [ ]:
y.shape

In [ ]:
y.plot(figsize=(10,6))
plt.show()

In [ ]:
#step 6: scalling
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

values = y.values.reshape(-1, 1)
scaled_values = scaler.fit_transform(values)

In [ ]:
#step 7 window creation
WINDOW_SIZE = 12   # it can be anything basically thisis how LSTM model learns

X = []  #X is basically the seq of past value
y = []  #y is basically value after those seq

for i in range(WINDOW_SIZE, len(scaled_values)):
    X.append(scaled_values[i-WINDOW_SIZE:i, 0])  # past 12 weeks
    y.append(scaled_values[i, 0])                # next week value

import numpy as np
X = np.array(X)
y = np.array(y)

# Reshape for LSTM: (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)


Creating train test split manually

In [ ]:
TEST_SIZE = 12  # last 12 weeks for testing

X_train = X[:-TEST_SIZE]
X_test  = X[-TEST_SIZE:]

y_train = y[:-TEST_SIZE]
y_test  = y[-TEST_SIZE:]



**Building the LSTM Model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)))
model.add(LSTM(32))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.summary()


In [ ]:
es = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    callbacks=[es],
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
y_pred_scaled = model.predict(X_test)


In [ ]:
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:
y_pred_scaled = model.predict(X_test)


In [ ]:
# Reshape because inverse_transform requires 2D
y_test_scaled_2d = y_test.reshape(-1, 1)
y_pred_scaled_2d = y_pred_scaled.reshape(-1, 1)

# Convert back to actual INR values
y_test_actual = scaler.inverse_transform(y_test_scaled_2d).flatten()
y_pred_actual = scaler.inverse_transform(y_pred_scaled_2d).flatten()


**SUMMARY**

Developed and trained an LSTM-based time series forecasting model on weekly and monthly spending data with full preprocessing, scaling, and sequence generation. The dataset showed irregular patterns with sudden spikes and no consistent trend or seasonality, making pattern learning challenging. Both time resolutions were impacted by this behavior. Proposed enhancements include exploring alternative models like Prophet for improved handling of irregular time-series data.

**Trying the Prophet Model**

In [ ]:
y_unscaled = df['amount_inr'].resample('MS').sum()  #prophet requires the unscalled value cause its not a neural network



In [ ]:
df_prophet = y_unscaled.reset_index()
df_prophet.columns = ['ds', 'y']


Due to environment issues with the original Prophet (Stan backend), I used NeuralProphet, a neural-network-based extension that follows the same additive forecasting philosophy as Prophet.

In [ ]:
!pip install neuralprophet


In [ ]:
from neuralprophet import NeuralProphet

model = NeuralProphet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
)

# Key: set learning_rate and disable checkpointing
metrics = model.fit(
    df_prophet,
    freq='MS',
    learning_rate=1e-3,      # any reasonable LR, e.g. 1e-3
    checkpointing=False,     # avoids the torch.load error
    progress='bar'           # optional, just for a nice progress bar
)

future = model.make_future_dataframe(df=df_prophet, periods=12)
forecast = model.predict(future)


In [ ]:
#plotting
model.plot(forecast)
model.plot_components(forecast)


The NeuralProphet model was trained on monthly spending data to forecast future expenses using a trend and seasonality decomposition approach similar to Prophet. Due to the highly irregular nature of the data—characterized by extended low-spending periods and sudden extreme spikes—the model faced challenges in learning consistent seasonal patterns. The trend component indicates a gradual decline in overall spending, while the seasonality component reflects irregular oscillations driven by the absence of true repeating cycles. These results highlight the difficulty of forecasting highly volatile, event-driven personal finance data and emphasize the importance of cautious interpretation and feature enrichment for future improvements.

**Summary**


Developed a complete time-series forecasting system on personal spending data using LSTM and Prophet-based models with preprocessing, smoothing, and sequence learning. The project explores trend, seasonality, and modeling challenges in highly irregular financial data.


In [ ]:
!pip install nbconvert


In [ ]:
!jupyter nbconvert --to notebook --ClearOutputPreprocessor.enabled=True --output clean.ipynb LSTM_spending_forecast.ipynb


In [ ]:
from google.colab import drive
drive.mount('/content/drive')




In [ ]:
!find /content -name "LSTM_spending_forecast.ipynb"


In [ ]:
!find /content -name "LSTM_spending_forecast.ipynb"
